In [ ]:
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
#https://chatgpt.com/c/6969200b-238c-8333-a6e9-70c8a88d2167
def loo_accuracy(X, y):
    loo = LeaveOneOut()
    preds = np.zeros_like(y)

    for tr, te in loo.split(X):
        clf = LogisticRegression(
            penalty='l2', C=1.0, solver='liblinear', max_iter=500
        )
        clf.fit(X[tr], y[tr])
        preds[te[0]] = clf.predict(X[te])[0]

    return (preds == y).mean()


vit = np.load('/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl', allow_pickle=True)['natural_scenes']
X   = np.load('/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy').T
y   = (np.argmax(vit,1) <= 397).astype(int)

def permuted_accuracy(seed):
    rng = np.random.RandomState(seed)
    y_perm = rng.permutation(y)
    return loo_accuracy(X, y_perm)

from joblib import Parallel, delayed

def permutation_test(X, y, n_perm=500, n_jobs=8):
    acc_real = loo_accuracy(X, y)

    acc_perm = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(permuted_accuracy)(seed)
        for seed in range(n_perm)
    )

    acc_perm = np.array(acc_perm)
    p_value = (acc_perm >= acc_real).mean()

    return acc_real, acc_perm, p_value

acc_real, acc_perm, p = permutation_test(X, y, n_perm=1000, n_jobs=12)

print("Real LOO accuracy:", acc_real)
print("Permutation p-value:", p)



[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   1 tasks      | elapsed:  1.8min
[Parallel(n_jobs=12)]: Done   8 tasks      | elapsed:  1.9min
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:  3.8min
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:  5.6min
[Parallel(n_jobs=12)]: Done  37 tasks      | elapsed:  7.5min
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:  8.0min
[Parallel(n_jobs=12)]: Done  61 tasks      | elapsed: 11.4min
[Parallel(n_jobs=12)]: Done  74 tasks      | elapsed: 13.4min
[Parallel(n_jobs=12)]: Done  89 tasks      | elapsed: 15.5min
[Parallel(n_jobs=12)]: Done 104 tasks      | elapsed: 17.5min
[Parallel(n_jobs=12)]: Done 121 tasks      | elapsed: 20.9min
[Parallel(n_jobs=12)]: Done 138 tasks      | elapsed: 23.3min
[Parallel(n_jobs=12)]: Done 157 tasks      | elapsed: 26.7min
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed: 29.1min
[Parallel(n_jobs=12)]: Done 197 tasks      | elapsed: 3

Real LOO accuracy: 0.711864406779661
Permutation p-value: 0.0


In [2]:
p

np.float64(0.0)